# 임베딩과 유사도 실전 (의미검색·중복탐지·코사인 vs 유클리드)
문장을 의미 벡터로 바꾸고, 코사인 유사도로 ① 같은 뜻인지 손으로 확인하고 ② 새 문의와 비슷한 과거 문의 top-k를 찾고 ③ 거의 같은 중복 문의 쌍을 찾아내고 ④ 코사인이 왜 유클리드보다 텍스트에 맞는지, ⑤ 정규화하면 코사인=내적이 됨까지 확인합니다.

### 0.준비

In [ ]:
# !pip install -q sentence-transformers pandas numpy   # 문장 임베딩 + 표/배열 라이브러리 설치(-q: 로그 최소화)

from google.colab import drive   # 코랩에서 구글 드라이브 연결용
drive.mount('/content/drive')   # 드라이브를 /content/drive 에 마운트(CSV 읽기 위함)

from pathlib import Path   # 경로를 / 로 안전하게 조합하는 도구
import numpy as np, pandas as pd   # 배열 계산(numpy) + 표 데이터(pandas)
from sentence_transformers import SentenceTransformer, util   # 임베딩 모델 + 유사도 도구(util)
ROOT = Path('/content/drive/MyDrive/kt cloud tech up/gen-ai'); DATA = ROOT / 'data'   # 데이터 폴더 경로 정의

model = SentenceTransformer('jhgan/ko-sroberta-multitask')   # 한국어 문장 임베딩 모델(문장→768차원 의미 벡터)
df = pd.read_csv(DATA / 'cs_inquiries.csv')   # 과거 CS 문의 280건 표로 읽기
texts = df['inquiry_text'].tolist()   # 문의 본문만 리스트로(임베딩 입력)

# encode: 280개 문의를 한 번에 의미 벡터로 변환
# convert_to_tensor=True : util.cos_sim에 바로 넣을 텐서 형태
emb = model.encode(texts, convert_to_tensor=True, show_progress_bar=True)
print('문의', len(texts), '건, 임베딩 shape', tuple(emb.shape))   # (280, 768): 문의 1개가 768차원 벡터 하나

Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.86k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

문의 280 건, 임베딩 shape (280, 768)


In [ ]:
emb[0,]

tensor([-2.4448e-01, -3.8816e-01,  4.9937e-01, -1.9208e-01, -5.4495e-01,
         2.6302e-03,  1.9086e-01, -5.6293e-01, -5.7794e-01,  6.4715e-01,
         2.6462e-01,  7.1111e-01, -3.7424e-01, -5.2403e-01,  1.6701e-01,
        -5.8633e-02,  7.0821e-01,  5.3986e-01, -3.1843e-01, -3.5252e-01,
        -2.4321e-02, -4.2658e-01,  5.1250e-01,  6.9606e-01, -5.5396e-01,
         1.9167e-01,  2.4233e-01, -7.8096e-01, -2.2348e-01, -7.8087e-01,
         2.7659e-01,  1.6099e-02,  3.1517e-02, -1.0839e-01,  1.1322e-01,
        -2.2702e-02, -2.6922e-01,  4.0429e-01, -4.5020e-01,  2.3787e-02,
        -9.6628e-02,  3.3337e-01, -2.8958e-01,  5.6159e-01,  4.4877e-01,
        -8.6290e-01,  2.8870e-01, -1.2841e-01, -7.8033e-01, -4.2805e-02,
        -2.7309e-02, -2.5397e-01, -1.2153e-01,  3.2441e-01, -1.2256e-01,
        -1.1791e-01, -1.6489e-01, -4.0074e-01,  1.5278e-01, -2.6945e-01,
        -3.6086e-01, -5.7866e-01,  4.3371e-02,  2.0973e-01, -7.0634e-01,
        -4.0596e-02, -2.5170e-02, -1.3947e-01, -8.8

### 1. 코사인 유사도 손으로 확인 — 글자가 안 겹쳐도 뜻이 가까우면 점수가 높다

In [ ]:
pairs = [('환불해 주세요', '돈 돌려받고 싶어요'),     # 같은 뜻, 글자는 안 겹침 → 높아야
         ('환불해 주세요', '배송 언제 오나요'),       # 뜻이 멀다 → 낮아야
         ('사이즈 교환하고 싶어요', '다른 치수로 바꿔주세요'),  # 같은 뜻(교환), 글자 거의 안 겹침 → 높아야
         ('회원 탈퇴할래요', '오늘 날씨 어때요')]      # 완전 무관 → 0 근처

for a, b in pairs:
    e = model.encode([a, b], convert_to_tensor=True)   # 두 문장을 같은 공간의 벡터로
    s = float(util.cos_sim(e[0], e[1]))   # 두 벡터의 방향이 얼마나 같은지(-1~1)
    print(f'{s:+.3f} | "{a}" ↔ "{b}"')

+0.620 | "환불해 주세요" ↔ "돈 돌려받고 싶어요"
+0.302 | "환불해 주세요" ↔ "배송 언제 오나요"
+0.656 | "사이즈 교환하고 싶어요" ↔ "다른 치수로 바꿔주세요"
+0.142 | "회원 탈퇴할래요" ↔ "오늘 날씨 어때요"


-> "환불"과 "돈 돌려받기"는 공통 글자가 하나도 없는데 0.62, "환불" ↔ "배송"은 0.30입니다. 글자가 아니라 뜻으로 점수가 갈리는 것이 임베딩의 핵심입니다. "회원 탈퇴" ↔ "날씨"처럼 완전히 무관하면 0.14로 0에 가깝게 떨어집니다.

### 2. 의미 검색 top-k — 새 문의와 비슷한 과거 문의 찾기

In [ ]:
def search(query, k=5):
    q = model.encode(query, convert_to_tensor=True)   # 질의도 같은 모델로 임베딩(같은 공간이라야 비교)
    top = util.cos_sim(q, emb)[0].topk(k)   # 모든 문의와의 코사인 → 상위 k개의 (값, 위치)
    return [(float(s), int(i)) for s, i in zip(top.values, top.indices)]

for query in ['주문한 옷 사이즈가 안 맞아요',          # '교환' 글자 없이도 교환 문의가 떠야
              '결제했는데 돈이 두 번 빠져나갔어요',     # '중복결제' 글자 없이도 결제 문의가 떠야
              '품절된 제품 다시 살 수 있나요']:         # '재입고' 글자 없이도 재입고 문의가 떠야
    print(f'질의: {query}')
    for s, i in search(query, k=3):
        print(f'  {s:.3f} [{df.iloc[i]["inquiry_type"]}] {texts[i]}')

질의: 주문한 옷 사이즈가 안 맞아요
  0.849 [교환] 옷 사이즈가 안 맞아서 다른 사이즈로요.
  0.644 [교환] 주문한 티셔츠 M 사이즈가 생각보다 너무 작네요. L 사이즈로 바꿔주실 수 있을까요?
  0.627 [취소] 옷을 주문했는데, 갑자기 살이 빠져서 이 사이즈는 너무 클 것 같아요. 새로 시켜야 할 것 같아서, 이 주문은 철회하고 싶어요.
질의: 결제했는데 돈이 두 번 빠져나갔어요
  0.718 [주문결제] 아까 스탠드 조명 결제했는데, 문자 메시지가 두 번 왔어요. 혹시 중복 결제된 건가요?
  0.713 [취소] 아, 제가 같은 상품을 두 번 결제했네요. [주문번호: 20240520-111222] 이거 하나만 배송하지 마시고 환불해주세요.
  0.591 [주문결제] 아까 결제 단계에서 창을 닫아버렸는데, 다시 이어서 결제할 수 있나요?
질의: 품절된 제품 다시 살 수 있나요
  0.736 [재입고] 품절된 실내화 핑크색 240 사이즈, 다시 판매할 계획이 있으신지 궁금합니다.
  0.729 [재입고] 전에 샀던 스탠드 조명 다시 사고 싶은데, 지금 품절이네요. 혹시 언제쯤 다시 구매할 수 있을까요?
  0.724 [재입고] 품절인 스킨케어 세트, 혹시 다른 구성으로라도 다시 나올까요?


### 3. 중복/유사 문의 탐지 — 거의 같은 문의 쌍 찾기(FAQ 통합용)

In [ ]:
sim = util.cos_sim(emb, emb).cpu().numpy()   # (280, 280) 모든 문의 쌍의 코사인 행렬
TH = 0.80   # 이 값 이상이면 '거의 같은 문의'로 간주
iu = np.triu_indices(len(texts), k=1)   # 위 삼각형(i<j)만 — 자기자신·중복쌍 제외
dup = sorted(((sim[i, j], i, j) for i, j in zip(*iu) if sim[i, j] >= TH), reverse=True)
print(f'유사 문의 쌍(코사인 ≥ {TH}): {len(dup)}쌍 — 상위 3')
for s, i, j in dup[:3]:
    print(f'{s:.3f}')
    print(f'  A {texts[i]}')
    print(f'  B {texts[j]}')

유사 문의 쌍(코사인 ≥ 0.8): 3쌍 — 상위 3
0.959
  A 생일 쿠폰은 언제 자동으로 지급되는 건가요? 제 생일이 이번 달인데 아직 안 들어왔어요.
  B 생일 쿠폰은 언제 지급되는 건가요? 이번 달이 제 생일인데 아직 안 들어왔어요. 혹시 조건이 있나요?
0.811
  A 로그인을 하려는데 계속 비밀번호가 틀리다고 나와요. 재설정 링크도 안 오고요.
  B 아이디랑 비밀번호를 정확히 입력했는데도 계속 '로그인 정보가 일치하지 않습니다'라고 떠요. 비밀번호 찾기 해도 등록된 정보가 없다고 하네요.
0.808
  A 바지 사이즈가 너무 커서 반품 처리했는데, 언제쯤 다시 돈이 들어올까요?
  B 사이즈가 안 맞아서 바지를 반품하고 싶은데, 반품할 때 배송비는 제가 부담해야 하는 건가요?


### 4. 유클리드 vs 코사인 — 코사인은 문장 길이에 안 휘둘린다

In [1]:
short = '환불해 주세요'
long_ = '제가 지난주에 주문한 상품을 환불받고 싶은데 절차가 어떻게 되는지 자세히 알려주실 수 있을까요'
other = '배송 언제 도착하나요'
v = model.encode([short, long_, other], convert_to_tensor=True).cpu().numpy()
euclid = lambda a, b: float(np.linalg.norm(a - b))   # 직선거리: 작을수록 비슷
cos    = lambda a, b: float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))   # 방향: 클수록 비슷
print(f'같은뜻(짧음↔길게): 유클리드 {euclid(v[0], v[1]):.3f} / 코사인 {cos(v[0], v[1]):.3f}')
print(f'다른뜻(환불↔배송): 유클리드 {euclid(v[0], v[2]):.3f} / 코사인 {cos(v[0], v[2]):.3f}')

같은뜻(짧음↔길게): 유클리드 9.508 / 코사인 0.616
다른뜻(환불↔배송): 유클리드 13.742 / 코사인 0.285


### 정규화 후 내적 = 코사인 — 벡터DB가 쓰는 속도 트릭
벡터를 모두 길이 1로 맞추면(정규화), 코사인 유사도가 그냥 내적(곱해서 더하기) 과 같아집니다. 나눗셈이 사라져 계산이 빨라집니다.

In [2]:
e2 = model.encode([short, long_, other], normalize_embeddings=True).astype('float32')  # 길이 1로 정규화
print('정규화 후 길이:', [round(float(np.linalg.norm(x)), 3) for x in e2])   # 모두 1.0
print(f'같은뜻 — 내적 {float(e2[0] @ e2[1]):.4f} vs 코사인 {cos(v[0], v[1]):.4f}')
print(f'다른뜻 — 내적 {float(e2[0] @ e2[2]):.4f} vs 코사인 {cos(v[0], v[2]):.4f}')

정규화 후 길이: [1.0, 1.0, 1.0]
같은뜻 — 내적 0.6162 vs 코사인 0.6162   → 같다
다른뜻 — 내적 0.2854 vs 코사인 0.2854   → 같다
